In [ ]:
!pip install -q "numpy==1.26.4" "deepface==0.0.93" "tf-keras"
import os
os.kill(os.getpid(), 9)

: 

In [4]:
import base64
import re
import numpy as np
import cv2
from IPython.display import display, Image as IPImage, Audio
from google.colab.output import eval_js
from PIL import Image
import io
import subprocess
from deepface import DeepFace

In [5]:
!wget -q -O piper.tar.gz https://github.com/rhasspy/piper/releases/download/2023.11.14-2/piper_linux_x86_64.tar.gz
!tar -xzf piper.tar.gz
!wget -q -O es_f.onnx https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_MX/claude/high/es_MX-claude-high.onnx
!wget -q -O es_f.onnx.json https://huggingface.co/rhasspy/piper-voices/resolve/main/es/es_MX/claude/high/es_MX-claude-high.onnx.json
print("✅ Piper voz femenina listo")


✅ Piper voz femenina listo


In [6]:
EMOCIONES = {
    'angry':    'enojo',
    'disgust':  'disgusto',
    'fear':     'miedo',
    'happy':    'felicidad',
    'sad':      'tristeza',
    'surprise': 'sorpresa',
    'neutral':  'neutral'
}

def age_range(age):
    if age < 18:  return "joven"
    if age < 35:  return "adulto joven"
    if age < 55:  return "adulto"
    return "adulto mayor"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_drive = '/content/drive/MyDrive/semillero-robotica-ia/deepface-analisis-imagen-emociones/foto-neutral.jpg'

frame_bgr = cv2.imread(path_drive)

if frame_bgr is not None:
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(frame_rgb))
else:
    print("❌ No se pudo encontrar la imagen. Verifica que la ruta sea correcta.")

In [ ]:
def analyze_face(img_bgr):
    print("🔍 Analizando con DeepFace...")
    try:
        results = DeepFace.analyze(
            img_path=img_bgr,
            actions=['age', 'gender', 'emotion'],
            enforce_detection=True,
            detector_backend='ssd',
            align=True
        )
        for i, face in enumerate(results):
            emo_es = EMOCIONES.get(face['dominant_emotion'], face['dominant_emotion'])
            rango  = age_range(face['age'])
            print(f"\n{'='*40}")
            print(f"  CARA #{i+1}")
            print(f"{'='*40}")
            print(f"  Edad estimada    : {face['age']} años ({rango})")
            print(f"  Género           : {face['dominant_gender']}")
            print(f"  Emoción          : {emo_es}")
            face['edad_rango'] = rango
            face['emocion_es'] = emo_es
        return results
    except ValueError as e:
        print(f"⚠️  No se detectó cara: {e}")
        return None

results = analyze_face(frame_bgr)

In [ ]:
results

In [ ]:
def draw_results(img_bgr, results):
    img_out = img_bgr.copy()
    for face in results:
        r = face['region']
        x, y, w, h = r['x'], r['y'], r['w'], r['h']
        label = f"{face['emocion_es']} | {face['edad_rango']} | {face['dominant_gender'][0]} | {face['age']}"
        cv2.rectangle(img_out, (x, y), (x+w, y+h), (0, 255, 100), 2)
        cv2.putText(img_out, label, (x, y - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 100), 2)
    return img_out

if results:
    annotated = draw_results(frame_bgr, results)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    pil_out = Image.fromarray(annotated_rgb)
    buf2 = io.BytesIO()
    pil_out.save(buf2, format='JPEG')
    display(IPImage(data=buf2.getvalue()))

In [ ]:
def speak(text):
    print(f"🔊 Sintetizando: {text}")
    proc = subprocess.run(
        ["./piper/piper", "--model", "es_f.onnx", "--output_file", "/tmp/out.wav"],
        input=text.encode("utf-8"),
        capture_output=True
    )
    if proc.returncode == 0:
        display(Audio("/tmp/out.wav", autoplay=True))
    else:
        print(f"⚠️  Error Piper: {proc.stderr.decode()}")


def build_speech(results):
    if not results:
        return "No detecté ningún rostro en la imagen."
    lines = []
    for face in results:
        gender_str = "hombre" if face['dominant_gender'] == "Man" else "mujer"
        lines.append(
            f"Detecto una persona. Aparenta ser un {gender_str} {face['edad_rango']}, "
            f"con expresión de {face['emocion_es']}. Bienvenido."
        )
    return " ".join(lines)


texto = build_speech(results)
print(f"\nTexto:\n{texto}\n")
speak(texto)

# Benchmarks - Comparacion de backends Deepface

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
path_drive = '/content/drive/MyDrive/semillero-robotica-ia/deepface-analisis-imagen-emociones/foto-neutral.jpg'

frame_bgr = cv2.imread(path_drive)

if frame_bgr is not None:
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    display(Image.fromarray(frame_rgb))
else:
    print("❌ No se pudo encontrar la imagen. Verifica que la ruta sea correcta.")

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
backends = [
    'opencv', 'ssd', 'dlib', 'mtcnn',
    'retinaface', 'mediapipe', 'yolov8n'
    , 'yunet', 'centerface',
]

import time
import pandas as pd

all_results = []

for b in backends:
  try:
    t0 = time.time()
    res = DeepFace.analyze(
            img_path=frame_bgr,
            actions=['age', 'gender', 'emotion'],
            detector_backend=b,
            enforce_detection=True,
            align=True,
          )
    elapsed_time = time.time() - t0

    # Extraemos datos del primer rostro detectado
    first_face = res[0]

    all_results.append({
        'backend': b,
        'tiempo_seg': round(elapsed_time, 3),
        'edad': first_face['age'],
        'genero': first_face['dominant_gender'],
        'emocion': first_face['dominant_emotion'],
        'confianza_cara': round(first_face['face_confidence'], 3)
    })

  except Exception as e:
    all_results.append({
        'backend': b,
        'tiempo_seg': round(time.time() - t0, 3),
        'edad': None,
        'genero': 'Error',
        'emocion': str(e)[:30],
        'confianza_cara': 0
    })

Action: emotion: 100%|██████████| 3/3 [00:00<00:00, 43.30it/s]


In [ ]:
df_comparativo = pd.DataFrame(all_results)
display(df_comparativo.sort_values(by='tiempo_seg'))